# BioTrace V2 - Ecosistema Multi-tenant y Object Storage

Este notebook demuestra las características Enterprise del proyecto:
1. **Multi-tenancy**: Aislamiento por clínica (Laboratorio).
2. **MinIO Object Storage**: Manejo de archivos físicos asociados a los análisis.
3. **Sistema de Alertas Automáticas**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from dao import AdminDAO, BioTraceDAO, StorageDAO

# 1. Usar AdminDAO para ver los tenants (Clínicas)
admin_dao = AdminDAO()
clinicas = list(admin_dao.col_clinicas.find())
admin_dao.cerrar_conexion()

if not clinicas:
    raise Exception("No hay clínicas. Ejecuta seed.py primero.")

print("Clínicas Disponibles (Tenants):")
for c in clinicas:
    print(f"- {c['nombre']} (ID: {c['_id']})")

# Elegimos la primera clínica para instanciar el DAO operativo
clinica_activa = clinicas[0]
dao = BioTraceDAO(clinica_id=clinica_activa['_id'])
print(f"\nDAO instanciado para: {clinica_activa['nombre']} - Asegurando aislamiento de datos.")

## 1. Verificación de Alertas Automáticas
Al insertar muestras con temperatura crítica (>4°C o <-85°C), el DAO genera una alerta.

In [ ]:
alertas = dao.listar_alertas()
print(f"Alertas Activas: {len(alertas)}")
for a in alertas:
    print(f"[! {a.tipo_alerta} !] {a.mensaje} (Fecha: {a.fecha_alerta})")

## 2. Lectura de Archivos desde Object Storage (MinIO)
Vamos a simular la recuperación de un archivo de secuenciación `.fastq` asociado a un paciente.

In [ ]:
pacientes = dao.listar_pacientes(limit=1)
if pacientes:
    p = pacientes[0]
    trazabilidad = dao.reporte_trazabilidad_paciente(p.id)
    storage = StorageDAO()
    
    print(f"Buscando archivos de análisis para paciente: {p.nombre} {p.apellido}\n")
    for muestra in trazabilidad:
        for analisis in muestra.get('analisis_realizados', []):
            path = analisis.get('resultados_crudos_path')
            if path:
                bucket, file = path.split('/', 1)
                print(f"Descargando '{file}' desde bucket '{bucket}'...")
                response = storage.client.get_object(bucket, file)
                contenido = response.read().decode('utf-8')
                response.close()
                response.release_conn()
                print(f"Contenido crudo:\n{contenido}")
                break
        break

## 3. Análisis Agrupados (Aggregation Pipeline)

In [ ]:
reporte_tipos = dao.reporte_analisis_por_tipo()
df_tipos = pd.DataFrame(reporte_tipos)
if not df_tipos.empty:
    df_tipos.rename(columns={'_id': 'Tipo de Análisis', 'total': 'Cantidad'}, inplace=True)
    plt.figure(figsize=(10, 5))
    plt.bar(df_tipos['Tipo de Análisis'], df_tipos['Cantidad'], color='teal')
    plt.title(f'Cantidad de Análisis por Tipo (Clínica: {clinica_activa["nombre"]})')
    plt.xlabel('Tipo de Análisis')
    plt.ylabel('Cantidad')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()